# MMALS Metric, Protocol and Claim Verification Stack v0.2.1.1

This notebook is reusable across RC2O, Geometry-MMALS, and TPUT. It does not
train a model or choose a route. It parses an execution PDF and a results ZIP,
then applies versioned metric, protocol, and claim rules.

Outputs:

```text
metric_report.json
protocol_report.json
claim_report.json
evidence_bundle.json
verification_summary.md
verification_gate_summary.csv
```

Statuses are restricted to `PASS`, `PARTIAL`, `FAIL`, and `NOT_TESTED`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"
EXPECTED_PACKAGE_VERSION = "1.1.1"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "pypdf>=4.0"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
import geometry_mmalls
assert geometry_mmalls.__version__ == EXPECTED_PACKAGE_VERSION
print("Verification source:", geometry_mmalls.__file__)


## 1. Declare inputs

Only these paths normally change between experiments. The output structure and
verification schemas remain the same.

In [ ]:
import json
import pandas as pd
import yaml
from geometry_mmalls.verification import verify_evidence_bundle, write_verification_outputs

EXPERIMENT_ID = os.environ.get("MMALS_EXPERIMENT_ID", "geometry_mmalls_g1_v1_1_0")
EXECUTION_PDF = Path(os.environ.get("MMALS_EXECUTION_PDF", "/content/drive/MyDrive/MMALS/Geometry_MMALS_G1_FunctionalRouting_v1_1_0.pdf"))
RESULTS_ZIP = Path(os.environ.get("MMALS_RESULTS_ZIP", "/content/drive/MyDrive/MMALS/Geometry_MMALS_G1_FunctionalRouting_v1_1_0_20260614_212819.zip"))
OUTPUT_DIR = Path(os.environ.get("MMALS_VERIFICATION_OUTPUT", f"/content/drive/MyDrive/MMALS/verification/{EXPERIMENT_ID}"))
METRIC_DEFINITIONS = REPO_DIR / "verification" / "metric_definitions.yaml"
PROTOCOL_RULES = REPO_DIR / "verification" / "protocol_rules.yaml"
CLAIM_RULES = REPO_DIR / "verification" / "claim_rules.yaml"
PROFILE_NAME = EXPERIMENT_ID.replace("geometry_mmalls_g1_", "geometry_g1_") + ".yaml"
EXPERIMENT_PROFILE = REPO_DIR / "verification" / "profiles" / PROFILE_NAME
assert EXPERIMENT_PROFILE.exists(), EXPERIMENT_PROFILE
for path in [EXECUTION_PDF, RESULTS_ZIP, METRIC_DEFINITIONS, PROTOCOL_RULES, CLAIM_RULES, EXPERIMENT_PROFILE]:
    assert path.exists(), path
print({"experiment_id": EXPERIMENT_ID, "pdf": str(EXECUTION_PDF), "zip": str(RESULTS_ZIP), "output": str(OUTPUT_DIR)})


## 2. Run the independent verifier

ZIP extraction is traversal-safe. The PDF is parsed with `pypdf`. Critical
protocol failures can invalidate positive metric results.

In [ ]:
bundle = verify_evidence_bundle(
    experiment_id=EXPERIMENT_ID,
    execution_pdf=EXECUTION_PDF,
    results_zip=RESULTS_ZIP,
    metric_definitions=METRIC_DEFINITIONS,
    protocol_rules=PROTOCOL_RULES,
    claim_rules=CLAIM_RULES,
    experiment_profile=EXPERIMENT_PROFILE,
)
outputs = write_verification_outputs(bundle, OUTPUT_DIR)
outputs


## 3. Inspect the standard gate table

In [ ]:
gate_table = pd.read_csv(outputs["verification_gate_summary"])
display(gate_table)
print(Path(outputs["verification_summary"]).read_text(encoding="utf-8"))


## 4. Hard-stop policy

The verifier refuses the bundle when any critical protocol rule fails.

In [ ]:
critical_failures = [
    item for item in bundle["protocol_report"]["rules"]
    if item["status"] == "FAIL" and item.get("evidence", {}).get("severity") == "critical"
]
if critical_failures:
    raise RuntimeError("Critical protocol verification failed: " + ", ".join(item["id"] for item in critical_failures))
print("No critical protocol failure.")


## 5. Red-team registry

In [ ]:
red_team = []
for path in sorted((REPO_DIR / "verification" / "red_team_cases").glob("*.yaml")):
    red_team.append(yaml.safe_load(path.read_text(encoding="utf-8")))
pd.DataFrame(red_team)


## 6. Portable evidence archive

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Verification bundle:", archive)
